In [27]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/nullset/Developer/ML/learning-project/notebooks


In [28]:
import sys, os
# Manually set your repo root (DO THIS ONCE)
repo_root = "/Users/nullset/Developer/ML/learning-project"

print("Repo root:", repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src import features

Repo root: /Users/nullset/Developer/ML/learning-project


In [29]:
raw_path = os.path.join(repo_root, "data", "raw")
interim_path = os.path.join(repo_root, "data", "interim")

features.process_folder(raw_path, interim_path, augment=True)

In [54]:
import os
import shutil
import random

processed_path = os.path.join(repo_root, "data", "processed")

def split_dataset(interim_dir, processed_dir, train_ratio=0.7, val_ratio=0.15):
    os.makedirs(processed_dir, exist_ok=True)
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(processed_dir, split), exist_ok=True)

    for class_name in os.listdir(interim_dir):
        class_path = os.path.join(interim_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f)) and f.lower().endswith(".npy")
        ]
        random.shuffle(files)

        n = len(files)
        if n == 0:
            continue

        n_train = int(train_ratio * n)
        n_val = int(val_ratio * n)

        splits = {
            "train": files[:n_train],
            "val": files[n_train:n_train+n_val],
            "test": files[n_train+n_val:]
        }

        for split, split_files in splits.items():
            split_class_dir = os.path.join(processed_dir, split, class_name)
            os.makedirs(split_class_dir, exist_ok=True)
            for f in split_files:
                src = os.path.join(class_path, f)
                dst = os.path.join(split_class_dir, f)
                shutil.copy(src, dst)

# Run it
split_dataset(interim_path, processed_path)
print("Dataset split complete")

Dataset split complete


In [55]:
import numpy as np

def load_npy_dataset(folder):
    X = []
    y = []

    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):
            if file.endswith(".npy"):
                X.append(np.load(os.path.join(class_path, file)))
                y.append(int(class_name))

    return np.array(X), np.array(y)

X_train, y_train = load_npy_dataset(os.path.join(processed_path, "train"))
X_val, y_val = load_npy_dataset(os.path.join(processed_path, "val"))

In [56]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(64,64,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes
])

In [57]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - accuracy: 0.2807 - loss: 1.2598 - val_accuracy: 0.3529 - val_loss: 1.0987
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.3158 - loss: 1.1285 - val_accuracy: 0.2941 - val_loss: 1.1195
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3333 - loss: 1.0872 - val_accuracy: 0.2941 - val_loss: 1.0946
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.3333 - loss: 1.0732 - val_accuracy: 0.2941 - val_loss: 1.0737
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3333 - loss: 1.0533 - val_accuracy: 0.2941 - val_loss: 1.0494
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3684 - loss: 1.0216 - val_accuracy: 0.4118 - val_loss: 1.0113
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5263 - loss: 0.9749 - val_accuracy: 0.4706 - val_loss: 0.9678
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5439 - loss: 0.9110 - val_accuracy: 0.6471 - val_loss: 0.8823

In [34]:
# Model shifted to models/

model.save(os.path.join(repo_root, "models", "hand_sign_cnn.h5"))

In [35]:
X_test, y_test = load_npy_dataset(os.path.join(processed_path, "test"))

model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6000 - loss: 0.9128


[0.9128161668777466, 0.6000000238418579]

In [53]:
sample = X_test[5]
pred = model.predict(sample.reshape(1,64,64,3))

print("Predicted:", np.argmax(pred))
print("Actual:", y_test[5])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Predicted: 1
Actual: 1
